<a href="https://colab.research.google.com/github/eric20041027/Data_Mining/blob/main/notebooks/train_pubmedbert_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PubMedBERT 5-Fold Training (Colab A100)

Trains `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` (renamed PubMedBERT) on the medical abstracts dataset, 5-fold StratifiedKFold, class-weighted CE, bf16.

**Before running:**
1. Runtime → Change runtime type → **A100 GPU**.
2. Run cells in order.
3. Optional: mount Drive at the end to back up `outputs/bert_runs/` before the session expires.

## 1. Clone repo

In [1]:
import os, sys, shutil
REPO_DIR = '/content/Data_Mining'
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone https://github.com/eric20041027/Data_Mining.git $REPO_DIR
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
# Persist HF cache on Drive if mounted later; for now use local Colab disk.
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('CWD:', os.getcwd())
print('Top-level:', sorted(os.listdir()))

Cloning into '/content/Data_Mining'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 32 (delta 5), reused 26 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 6.26 MiB | 20.53 MiB/s, done.
Resolving deltas: 100% (5/5), done.
CWD: /content/Data_Mining
Top-level: ['.git', '.gitignore', 'README.md', 'Rule.md', 'kaggle_testset.csv', 'kaggle_testset_submission.csv', 'kaggle_trainset.csv', 'notebooks', 'outputs', 'plan.md', 'src']


## 2. Install dependencies

In [2]:
!pip install -q -U "transformers>=4.44,<4.50" "accelerate>=0.33" "datasets>=2.20" "scikit-learn>=1.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.0 MB/s eta 0:00:00


In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
else:
    print('WARNING: no GPU detected — change runtime to A100 before training.')

CUDA available: True
Device: NVIDIA A100-SXM4-40GB
bf16 supported: True


## 3. Smoke test — 1 fold × 1 epoch on 64 samples
Confirms the pipeline runs end-to-end before launching the full sweep (~30 sec on A100).

In [4]:
!python src/train_bert.py --fold 0 --seed 42 --smoke --tag smoke_test

2026-05-21 01:41:06.814432: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-21 01:41:06.886554: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 28.0/28.0 [00:00<00:00, 255kB/s]
config.json: 100% 385/385 [00:00<00:00, 4.04MB/s]
vocab.txt: 226kB [00:00, 12.7MB/s]
pytorch_model.bin: 100% 440M/440M [00:02<00:00, 156MB/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly

## 4. Full 5-fold sweep — PubMedBERT base, seed=42
Each fold ≈ 8–12 min on A100 → total ~50 min.

In [5]:
MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SEED = 42
for fold in range(5):
    cmd = (
        f'python src/train_bert.py '
        f'--model {MODEL} --fold {fold} --seed {SEED} '
        f'--epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 '
        f'--tag pubmedbert_base_seed{SEED}_fold{fold}'
    )
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'fold {fold} failed'

>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 0 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed42_fold0
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 1 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed42_fold1
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 2 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed42_fold2
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 3 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed42_fold3
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 4 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 

## 5. Summarise OOF Macro F1 across folds

In [6]:
import json, glob, numpy as np, pandas as pd
runs = sorted(glob.glob('outputs/bert_runs/pubmedbert_base_seed42_fold*/metrics.json'))
rows = [json.load(open(p)) for p in runs]
df = pd.DataFrame(rows)
print(df[['fold', 'val_macro_f1', 'train_secs']])
print(f'\nMean OOF Macro F1: {df.val_macro_f1.mean():.4f} (std {df.val_macro_f1.std():.4f})')

   fold  val_macro_f1  train_secs
0     0      0.649675  165.787410
1     1      0.640046  165.858974
2     2      0.634988  165.273319
3     3      0.646403  165.908564
4     4      0.629006  165.204920

Mean OOF Macro F1: 0.6400 (std 0.0084)


## 6. Build final submission (BERT + overlap constraint)

In [7]:
!python src/ensemble_predict.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_base_seed42_fold*' \
    --tag pubmedbert_v1
print()
!ls -la outputs/submission_*.csv

Found 5 BERT run dirs:
  outputs/bert_runs/pubmedbert_base_seed42_fold0
  outputs/bert_runs/pubmedbert_base_seed42_fold1
  outputs/bert_runs/pubmedbert_base_seed42_fold2
  outputs/bert_runs/pubmedbert_base_seed42_fold3
  outputs/bert_runs/pubmedbert_base_seed42_fold4
  fold 0: 1 run(s), val macro F1 = 0.6497
  fold 1: 1 run(s), val macro F1 = 0.6400
  fold 2: 1 run(s), val macro F1 = 0.6350
  fold 3: 1 run(s), val macro F1 = 0.6464
  fold 4: 1 run(s), val macro F1 = 0.6290

BERT OOF Macro F1: 0.6402
                                 precision    recall  f1-score   support

                      neoplasms     0.7214    0.7987    0.7581      2837
      digestive system diseases     0.5175    0.8110    0.6318      1333
        nervous system diseases     0.5208    0.7751    0.6230      1743
        cardiovascular diseases     0.6807    0.8507    0.7563      2747
general pathological conditions     0.7542    0.3023    0.4316      4334

                       accuracy                        

## 7. Back up artefacts before session ends

**Option A — download a tarball locally:**

In [8]:
!tar -czf bert_runs.tar.gz outputs/bert_runs outputs/submission_pubmedbert_v1.csv
from google.colab import files
files.download('bert_runs.tar.gz')
files.download('outputs/submission_pubmedbert_v1.csv')

^C


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Option B — copy to Google Drive:**

In [9]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Kaggle_new_backup
!cp -r outputs/bert_runs /content/drive/MyDrive/Kaggle_new_backup/
!cp outputs/submission_pubmedbert_v1.csv /content/drive/MyDrive/Kaggle_new_backup/

Mounted at /content/drive


## 8. Optional: extra seeds for ensemble
Uncomment to run additional seeds. Each adds ~50 min.

In [ ]:
# MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
# for SEED in (2024, 7):
#     for fold in range(5):
#         cmd = (
#             f'python src/train_bert.py '
#             f'--model {MODEL} --fold {fold} --seed {SEED} '
#             f'--epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 '
#             f'--tag pubmedbert_base_seed{SEED}_fold{fold}'
#         )
#         print('>>>', cmd)
#         rc = os.system(cmd)
#         assert rc == 0, f'seed {SEED} fold {fold} failed'

In [10]:
# 拉最新的校正腳本
!cd /content/Data_Mining && git pull

# 跑校正
!cd /content/Data_Mining && python src/calibrate_and_submit.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_base_seed42_fold*' \
    --tag pubmedbert_v2_calibrated

# 下載新的 submission
from google.colab import files
files.download('/content/Data_Mining/outputs/submission_pubmedbert_v2_calibrated.csv')

remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 6), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 5.77 KiB | 1.92 MiB/s, done.
From https://github.com/eric20041027/Data_Mining
   ec49215..969723d  main       -> origin/main
Updating ec49215..969723d
Fast-forward
 notebooks/train_pubmedbert_colab.ipynb | 117 ++++++++++++++----
 src/calibrate_and_submit.py            | 211 +++++++++++++++++++++++++++++++++
 2 files changed, 305 insertions(+), 23 deletions(-)
 create mode 100644 src/calibrate_and_submit.py
  fold 0: 1 run(s)
  fold 1: 1 run(s)
  fold 2: 1 run(s)
  fold 3: 1 run(s)
  fold 4: 1 run(s)

Uncalibrated OOF Macro F1: 0.6402

Coarse search...
  start macro F1 = 0.6402
  round 0: macro F1 = 0.6498, bias = [-0.8, -0.6, -0.8, -0.5, 0.0]
  round 1: macro F1 = 0.6498, bias = [-0.8, -0.6, -0.8, -0.5, 0.0]

Fine refinement (anchored to coarse solutio

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SEED = 2024
for fold in range(5):
    cmd = (f'python src/train_bert.py --model {MODEL} '
           f'--fold {fold} --seed {SEED} --epochs 4 --batch-size 32 '
           f'--lr 2e-5 --max-length 512 '
           f'--tag pubmedbert_base_seed{SEED}_fold{fold}')
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'fold {fold} failed'

In [ ]:
!cd /content/Data_Mining && python src/calibrate_and_submit.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_base_seed*_fold*' \
    --tag pubmedbert_v4_2seeds

In [14]:
!cd /content/Data_Mining && git pull
!cd /content/Data_Mining && python src/calibrate_and_submit.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_base_seed42_fold*' \
    --no-overlap-constraint \
    --tag pubmedbert_v5_calibrated_no_overlap

from google.colab import files
files.download('/content/Data_Mining/outputs/submission_pubmedbert_v5_calibrated_no_overlap.csv')

Already up to date.
  fold 0: 1 run(s)
  fold 1: 1 run(s)
  fold 2: 1 run(s)
  fold 3: 1 run(s)
  fold 4: 1 run(s)

Uncalibrated OOF Macro F1: 0.6402

Coarse search...
  start macro F1 = 0.6402
  round 0: macro F1 = 0.6498, bias = [-0.8, -0.6, -0.8, -0.5, 0.0]
  round 1: macro F1 = 0.6498, bias = [-0.8, -0.6, -0.8, -0.5, 0.0]

Fine refinement (anchored to coarse solution)...
  fine round 0: F1=0.6500, bias=[-0.79, -0.99, -0.8, -0.48, 0.0]
  fine round 1: F1=0.6503, bias=[-0.89, -1.04, -0.8, -0.48, 0.0]
  fine round 2: F1=0.6503, bias=[-0.89, -1.04, -0.8, -0.48, 0.0]

Calibrated OOF Macro F1: 0.6503
                                 precision    recall  f1-score   support

                      neoplasms     0.7380    0.7674    0.7524      2837
      digestive system diseases     0.5463    0.7344    0.6266      1333
        nervous system diseases     0.5452    0.7126    0.6178      1743
        cardiovascular diseases     0.6868    0.8311    0.7521      2747
general pathological conditi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import os
MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SEED = 2024
for fold in range(5):
    cmd = (f'python src/train_bert.py --model {MODEL} '
           f'--fold {fold} --seed {SEED} --epochs 4 --batch-size 32 '
           f'--lr 2e-5 --max-length 512 '
           f'--tag pubmedbert_base_seed{SEED}_fold{fold}')
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'fold {fold} failed'

# 2-seed ensemble submission
!cd /content/Data_Mining && python src/ensemble_predict.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_base_seed*_fold*' \
    --no-overlap-constraint \
    --tag pubmedbert_v7_2seeds

from google.colab import files
files.download('/content/Data_Mining/outputs/submission_pubmedbert_v7_2seeds.csv')

>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 0 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed2024_fold0
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 1 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed2024_fold1
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 2 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed2024_fold2
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 3 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed2024_fold3
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 4 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Kaggle_backup', exist_ok=True)
!tar -czf /content/drive/MyDrive/Kaggle_backup/pubmedbert_2seeds.tar.gz \
    -C /content/Data_Mining outputs/bert_runs

!ls -lh /content/drive/MyDrive/Kaggle_backup/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 12G
-rw------- 1 root root 12G May 21 03:31 pubmedbert_2seeds.tar.gz


In [17]:
import os
MODEL = 'dmis-lab/biobert-base-cased-v1.2'
SEED = 42
for fold in range(5):
    cmd = (f'python src/train_bert.py --model {MODEL} '
           f'--fold {fold} --seed {SEED} --epochs 4 --batch-size 32 '
           f'--lr 2e-5 --max-length 512 '
           f'--tag biobert_base_seed{SEED}_fold{fold}')
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'fold {fold} failed'

# 印每 fold 的 val macro F1 摘要
import json, glob, pandas as pd
runs = sorted(glob.glob('outputs/bert_runs/biobert_base_seed42_fold*/metrics.json'))
df = pd.DataFrame([json.load(open(p)) for p in runs])
print(df[['fold', 'val_macro_f1', 'train_secs']])
print(f'BioBERT mean OOF Macro F1: {df.val_macro_f1.mean():.4f}')

>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 0 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag biobert_base_seed42_fold0
>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 1 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag biobert_base_seed42_fold1
>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 2 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag biobert_base_seed42_fold2
>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 3 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag biobert_base_seed42_fold3
>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 4 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag biobert_base_seed42_fold4
   fold  val_macro_f1  train_secs
0     0      0.657798  178.605288
1     1      0.644369  177.961635
2     2      0.626802  

In [18]:
# 候選 A: PubMedBERT × 2 + BioBERT × 1
!cd /content/Data_Mining && python src/ensemble_predict.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_base_seed*_fold*' \
                'outputs/bert_runs/biobert_base_seed*_fold*' \
    --no-overlap-constraint \
    --tag pubmedbert_v9_2seeds_plus_biobert

# 候選 B: BioBERT only（看純 BioBERT 強度）
!cd /content/Data_Mining && python src/ensemble_predict.py \
    --bert-runs 'outputs/bert_runs/biobert_base_seed*_fold*' \
    --no-overlap-constraint \
    --tag biobert_only

# 備份這次新增的 runs + submissions
!tar -czf /content/drive/MyDrive/Kaggle_backup/all_runs_$(date +%Y%m%d).tar.gz \
    -C /content/Data_Mining outputs/bert_runs outputs/submission_pubmedbert_v9_2seeds_plus_biobert.csv outputs/submission_biobert_only.csv

from google.colab import files
files.download('/content/Data_Mining/outputs/submission_pubmedbert_v9_2seeds_plus_biobert.csv')
files.download('/content/Data_Mining/outputs/submission_biobert_only.csv')

Found 15 BERT run dirs:
  outputs/bert_runs/biobert_base_seed42_fold0
  outputs/bert_runs/biobert_base_seed42_fold1
  outputs/bert_runs/biobert_base_seed42_fold2
  outputs/bert_runs/biobert_base_seed42_fold3
  outputs/bert_runs/biobert_base_seed42_fold4
  outputs/bert_runs/pubmedbert_base_seed2024_fold0
  outputs/bert_runs/pubmedbert_base_seed2024_fold1
  outputs/bert_runs/pubmedbert_base_seed2024_fold2
  outputs/bert_runs/pubmedbert_base_seed2024_fold3
  outputs/bert_runs/pubmedbert_base_seed2024_fold4
  outputs/bert_runs/pubmedbert_base_seed42_fold0
  outputs/bert_runs/pubmedbert_base_seed42_fold1
  outputs/bert_runs/pubmedbert_base_seed42_fold2
  outputs/bert_runs/pubmedbert_base_seed42_fold3
  outputs/bert_runs/pubmedbert_base_seed42_fold4
  fold 0: 3 run(s), val macro F1 = 0.6595
  fold 1: 3 run(s), val macro F1 = 0.6494
  fold 2: 3 run(s), val macro F1 = 0.6387
  fold 3: 3 run(s), val macro F1 = 0.6520
  fold 4: 3 run(s), val macro F1 = 0.6342

BERT OOF Macro F1: 0.6469
         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import os, tarfile, glob

src = '/content/Data_Mining/outputs/bert_runs'
dst = '/content/drive/MyDrive/Kaggle_backup/predictions_only_before_noweight.tar.gz'
os.makedirs(os.path.dirname(dst), exist_ok=True)

patterns = ['*.npy', '*.json', '*.csv']
files_to_pack = []
for run in sorted(os.listdir(src)):
    run_dir = os.path.join(src, run)
    if not os.path.isdir(run_dir):
        continue
    for pat in patterns:
        files_to_pack += glob.glob(os.path.join(run_dir, pat))

print(f'Packing {len(files_to_pack)} files...')
with tarfile.open(dst, 'w:gz') as tar:
    for f in files_to_pack:
        tar.add(f, arcname=os.path.relpath(f, '/content/Data_Mining'))

import subprocess
size = subprocess.check_output(['du', '-h', dst]).decode().split()[0]
print(f'Backup OK: {dst} ({size})')

Packing 80 files...
Backup OK: /content/drive/MyDrive/Kaggle_backup/predictions_only_before_noweight.tar.gz (1.2M)


In [20]:
import os
MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SEED = 42
for fold in range(5):
    cmd = (f'python src/train_bert.py --model {MODEL} '
           f'--fold {fold} --seed {SEED} --epochs 4 --batch-size 32 '
           f'--lr 2e-5 --max-length 512 '
           f'--class-weight none '
           f'--tag pubmedbert_noweight_seed{SEED}_fold{fold}')
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'fold {fold} failed'

# 印 OOF 摘要
import json, glob
import pandas as pd
runs = sorted(glob.glob('outputs/bert_runs/pubmedbert_noweight_seed42_fold*/metrics.json'))
df = pd.DataFrame([json.load(open(p)) for p in runs])
print(df[['fold', 'val_macro_f1', 'train_secs']])
print(f'\nNo-weight mean OOF Macro F1: {df.val_macro_f1.mean():.4f}')

>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 0 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --class-weight none --tag pubmedbert_noweight_seed42_fold0
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 1 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --class-weight none --tag pubmedbert_noweight_seed42_fold1
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 2 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --class-weight none --tag pubmedbert_noweight_seed42_fold2
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 3 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --class-weight none --tag pubmedbert_noweight_seed42_fold3
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-unc

In [ ]:
# A. noweight only (純 noweight 5-fold)
!cd /content/Data_Mining && python src/ensemble_predict.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_noweight_seed42_fold*' \
    --no-overlap-constraint \
    --tag pubmedbert_noweight_only

# B. v10 = v9 + noweight (4 模型 ensemble)
!cd /content/Data_Mining && python src/ensemble_predict.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_base_seed*_fold*' \
                'outputs/bert_runs/biobert_base_seed*_fold*' \
                'outputs/bert_runs/pubmedbert_noweight_seed*_fold*' \
    --no-overlap-constraint \
    --tag v10_4models

# C. v11 = noweight + BioBERT (兩個沒受 balanced 影響的模型)
#    注意 BioBERT 之前是用 balanced 訓的, 所以這個是 noweight + balanced biobert
!cd /content/Data_Mining && python src/ensemble_predict.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_noweight_seed*_fold*' \
                'outputs/bert_runs/biobert_base_seed*_fold*' \
    --no-overlap-constraint \
    --tag v11_noweight_plus_biobert

# 列出所有產生的 submission
!ls -lh outputs/submission_*.csv

# 下載
from google.colab import files
files.download('/content/Data_Mining/outputs/submission_pubmedbert_noweight_only.csv')
files.download('/content/Data_Mining/outputs/submission_v10_4models.csv')
files.download('/content/Data_Mining/outputs/submission_v11_noweight_plus_biobert.csv')

# 再備份一次（含新的 runs）
import tarfile
src = '/content/Data_Mining/outputs/bert_runs'
dst = '/content/drive/MyDrive/Kaggle_backup/predictions_only_after_noweight.tar.gz'
patterns = ['*.npy', '*.json', '*.csv']
files_to_pack = []
for run in sorted(os.listdir(src)):
    run_dir = os.path.join(src, run)
    if not os.path.isdir(run_dir): continue
    for pat in patterns:
        files_to_pack += glob.glob(os.path.join(run_dir, pat))
with tarfile.open(dst, 'w:gz') as tar:
    for f in files_to_pack:
        tar.add(f, arcname=os.path.relpath(f, '/content/Data_Mining'))
print(f'Final backup: {dst}')